<a href="https://colab.research.google.com/github/google-ai-edge/litert-samples/blob/main/benchmark/developer_device_platform/ddp_benchmark_advanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2026 The AI Edge Authors.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ==============================================================================

# Advanced: benchmarking on DDP with the `device-run` CLI

> Looking for the simple path? [`ddp_benchmark.ipynb`](ddp_benchmark.ipynb) benchmarks a single model on a single device with one `litert benchmark --ddp` command. **This** notebook drops down to the underlying `gcloud alpha device-run` CLI, which is what you need for multi-device sweeps, custom binaries, and full control over benchmark flags.

This notebook runs [`litert_lm_advanced_main`](https://github.com/google-ai-edge/LiteRT-LM/blob/main/runtime/engine/litert_lm_advanced_main.cc)
against **Gemma 4 E2B** on physical Android phones, using
**Developer Device Platform (DDP)** to allocate the hardware.

It demonstrates **two ways to call DDP**:

| Execution Method | Mechanism | Ideal Use Case |
| :--- | :--- | :--- |
| **Part 1:<br>Direct Execution** | Directly invokes the benchmark binary<br>with a fixed set of arguments for a<br>single backend and device. | Quick, baseline evaluations<br>requiring only basic<br>command-line flags. |
| **Part 2:<br>Wrapper Script** | Deploys a custom shell script to<br>orchestrate multiple configurations<br>(e.g., CPU and GPU) on each device. | Comprehensive benchmarking<br>requiring cache warm-ups,<br>and thermal management. |




## 🛠️ 1. Environment setup

In [ ]:
# @title Set up your GCP project for DDP
# @markdown Set your GCP project id (DDP sessions are billed to it). This cell
# @markdown logs in to gcloud and enables the Device Run API.

ddp_gcp_project = "your-own-gcp-project-id" # @param {type:"string"}

# @markdown The GCS bucket DDP uses for inputs and results. Leave blank to use
# @markdown the default `{project_id}-devicerun`.
custom_bucket_id = "" # @param {type:"string"}

bucket_id = custom_bucket_id if custom_bucket_id else f"{ddp_gcp_project}-devicerun"

if not ddp_gcp_project or ddp_gcp_project == "your-own-gcp-project-id":
  raise ValueError("Please specify a valid GCP Project ID in the form above.")

# Login to gcloud and set Application Default Credentials via Colab Auth
from google.colab import auth
auth.authenticate_user()

# Enable the devicerun API
!gcloud services enable devicerun.googleapis.com --project {ddp_gcp_project}

# DDP stages inputs and writes results here; create it if it is missing.
!gcloud storage buckets describe "gs://{bucket_id}" >/dev/null 2>&1 \
  || gcloud storage buckets create "gs://{bucket_id}" --project={ddp_gcp_project}

print(f"Project: {ddp_gcp_project}\nBucket:  gs://{bucket_id}")


In [ ]:
# @title Download latest version of GCloud CLI
# @markdown This ensures the latest features from DDP CLI are available in the colab.

import os

# 1. Download and extract the absolute latest gcloud release directly to /opt
!wget -qO- https://dl.google.com/dl/cloudsdk/channels/rapid/downloads/google-cloud-cli-linux-x86_64.tar.gz | tar xz -C /opt

# 2. Run the silent install script
!/opt/google-cloud-sdk/install.sh -q

# 3. Add the new installation to the front of the Python environment's PATH
os.environ["PATH"] = f"/opt/google-cloud-sdk/bin:{os.environ['PATH']}"

# 4. Explicitly install the alpha components using the newly downloaded CLI
!gcloud components install alpha --quiet


## 📦 2. Binaries and Model

**The Binaries** are hosted in the public LiteRT release bucket on Google Cloud Storage under `gs://litert/binaries/latest/android_arm64/litert_lm/`.

To run the benchmark on a physical Android device, we need the main executable along with its specific shared library (`.so`) dependencies:

| File | Purpose |
| :--- | :--- |
| `litert_lm_advanced_main` | The core benchmark executable. |
| `libGemmaModelConstraintProvider.so` | **Required.** A link-time (`DT_NEEDED`) dependency for Gemma models. |
| `libLiteRtOpenClAccelerator.so` | **GPU only.** The OpenCL delegate library, dynamically loaded at runtime when `--backend=gpu` is passed. |
| `libLiteRtTopKOpenClSampler.so` | **GPU only.** The OpenCL sampler library, also dynamically loaded for GPU inference. |


**A note on execution:** Because the two OpenCL libraries are opened at *runtime* (`dlopen`) rather than linked at compile time, the Android loader must be explicitly told where to find them. This is why our shell script exports `LD_LIBRARY_PATH=/data/local/tmp` before running the binary!

**The Model**: For this colab, we use [Gemma4 E2B](https://huggingface.co/litert-community/gemma-4-E2B-it-litert-lm). It is pulled straight from Hugging Face.


In [ ]:
# @title Stage the model in your GCS bucket
# @markdown Downloads the model from HuggingFace and copies it to your bucket.
# @markdown DDP pushes files to devices from GCS, so it has to live there first.
# @markdown
# @markdown This is a ~2.6 GB transfer on the first run. The cell skips the work
# @markdown if the object is already in the bucket.
import subprocess

# Gemma4 E2B model on HF
HF_REPO = "litert-community/gemma-4-E2B-it-litert-lm"
HF_FILE = "gemma-4-E2B-it.litertlm"
# Model location in GCS
MODEL_URI = f"gs://{bucket_id}/models/{HF_FILE}"

# Check if the model already exists, to avoid a 2.6 GB download + upload cycle.
exists = subprocess.run(["gcloud", "storage", "ls", MODEL_URI],
                        capture_output=True).returncode == 0

if exists:
  print(f"Already staged, skipping download: {MODEL_URI}")
else:
  url = f"https://huggingface.co/{HF_REPO}/resolve/main/{HF_FILE}"
  print(f"Downloading {url}")
  !curl -L --fail -o "{HF_FILE}" "{url}"
  !gcloud storage cp "{HF_FILE}" "{MODEL_URI}"
  # Free the Colab disk; the copy in GCS is the one that matters.
  !rm -f "{HF_FILE}"

print(f"Model: {MODEL_URI}")

# Public LiteRT release bucket. Use the copy inside `litert_lm/`: the binary one
# directory up is an older build and does not match these libraries.
LITERT_LM = "gs://litert/binaries/latest/android_arm64/litert_lm"
LM_BIN = f"{LITERT_LM}/litert_lm_advanced_main"

WORK = "/data/local/tmp"  # where DDP pushes files on the device

# Libraries that must sit next to the binary on the device.
DEVICE_LIBS = [
    "libGemmaModelConstraintProvider.so",
    "libLiteRtOpenClAccelerator.so",
    "libLiteRtTopKOpenClSampler.so",
]
LIB_PUSH = [f"{LITERT_LM}/{lib}={WORK}/{lib}" for lib in DEVICE_LIBS]


## 📱 3. Finding device IDs

Both parts below refer to devices by their DDP catalog ID (`pa3q-35`,
`caiman-35`, …). To see what your project can allocate, run the **Catalog API**
in a scratch cell:

```bash
!gcloud alpha device-run devices list \
  --project="{ddp_gcp_project}" --location="global" --format="json"
```

Each entry carries an `availability` field — prefer devices marked `HIGH` to
keep queue time short. Note that availability is per *(model, OS version)*
pair, so `pa3q-35` and `pa3q-36` can differ.

The IDs you pick feed `DIRECT_DEVICE` in Part 1 and `ddp_target_devices` in
Part 2.


## 🎯 Part 1 — The Direct Call

The simplest way to use the Developer Device Platform (DDP) is to pass it a raw C++ binary, execute it on a physical device, and read back the output.

```bash
gcloud alpha device-run sessions submit android-executable \
  --executable=gs://.../litert_lm_advanced_main \
  --executable-args="--model_path=...,--backend=gpu,..." \
  --executable-env-vars="LD_LIBRARY_PATH=/data/local/tmp" \
  --other-files-to-push="...=/data/local/tmp/model.litertlm"
```

In this approach, three specific flags orchestrate the entire execution:

* **`--executable`**: The compiled C++ binary itself. No wrapper shell script is used.
* **`--executable-args`**: A *comma-separated* list of arguments passed directly to the binary. *(Note: If an argument's value naturally contains a comma, you must escape it using the `^SEP^` syntax).*
* **`--executable-env-vars`**: Because there is no shell environment to run `export LD_LIBRARY_PATH`, we inject environment variables directly through the API. This is crucial for GPU runs, as the binary needs to know where to dynamically load the OpenCL delegate (`.so`) we pushed to the device.

### 📝 How Output is Captured
DDP automatically captures the Android device's system log (**Logcat**) and uploads it to your Cloud Storage bucket as `logcat.txt`. This file is exactly where we will parse our benchmark metrics in the next step!


### ⚖️ The Trade-off
This approach is incredibly fast and simple to set up, but **one session runs exactly one configuration.**

Because `--executable-args` is a session-level flag, every device in the test pool receives the exact same parameters. To test multiple backends (like CPU *and* GPU), you would need to submit multiple sessions or use a wrapper script (which we cover in Part 2).

In [ ]:
# @title Run a single benchmark directly
# @markdown One device, one backend, no shell script.

DIRECT_DEVICE  = "pa3q-35" # @param {type:"string"}
DIRECT_BACKEND = "gpu" # @param ["cpu", "gpu"]

PREFILL_TOKENS = 1024 # @param {type:"integer"}
DECODE_TOKENS  = 512 # @param {type:"integer"}
MAX_NUM_TOKENS = 2048 # @param {type:"integer"}
CPU_THREADS    = 4 # @param {type:"integer"}

# LM_BIN, LIB_PUSH, WORK and MODEL_URI all come from the setup section above.
args = [
    f"--model_path={WORK}/model.litertlm",
    "--benchmark=true",
    f"--benchmark_prefill_tokens={PREFILL_TOKENS}",
    f"--benchmark_decode_tokens={DECODE_TOKENS}",
    f"--max_num_tokens={MAX_NUM_TOKENS}",
    "--report_peak_memory_footprint=true",
    f"--backend={DIRECT_BACKEND}",
]
# On GPU the driver picks its own thread count, so the flag is CPU-only.
if DIRECT_BACKEND == "cpu":
  args.append(f"--num_cpu_threads={CPU_THREADS}")

push = [f"{MODEL_URI}={WORK}/model.litertlm"] + LIB_PUSH

direct_output = !gcloud alpha device-run sessions submit android-executable \
  --project="{ddp_gcp_project}" \
  --location="global" \
  --bucket-name="{bucket_id}" \
  --device="{DIRECT_DEVICE}" \
  --executable="{LM_BIN}" \
  --executable-args="{','.join(args)}" \
  --executable-env-vars="LD_LIBRARY_PATH={WORK}" \
  --other-files-to-push="{','.join(push)}" \
  --executable-timeout=30m 2>&1

print("\n".join(direct_output))


In [ ]:
# @title Adds some methods to parse the metrics
import re
import glob
import pandas as pd
from pathlib import Path

# Both halves of this cell must agree on where session artifacts land:
# download_session_results() writes here, and the parsing helpers read
# from here. Keeping it in one constant stops them drifting apart.
RESULTS_DIR = "/content/ddp_results"

def parse_benchmark_metrics(filepath: str) -> dict:
    """
    Parses LiteRT-LM benchmark metrics from a filepath.

    Args:
      session_id: The ID of the DeviceRun session to parse.

    Returns:
      A dictionary containing the parsed metrics (prefill, decode, TTFT, peak RAM, GPU status).
    """
    # Read the log file, using 'replace' to safely handle any invalid characters from device logs.
    with open(filepath, "r", errors="replace") as f:
        body = f.read()

    # Define regex patterns to extract the required metrics.
    # Metrics are printed once per inference turn; we will take the last occurrence.
    patterns = {
        "prefill_tok_s": r"Prefill Speed:\s*([\d.]+) tokens/sec",
        "decode_tok_s": r"Decode Speed:\s*([\d.]+) tokens/sec",
        "ttft_s": r"Time to first token:\s*([\d.]+) s",
        "peak_ram_mb": r"Peak system ram usage:\s*([\d.]+)",
    }

    metrics = {}
    for key, pat in patterns.items():
        # Find all matches for the current pattern in the log body
        found = re.findall(pat, body)
        # If matches are found, take the last one (the most recent turn) and convert to float
        metrics[key] = float(found[-1]) if found else None

    # Check if the OpenCL delegate marker is present, which indicates successful GPU delegation
    metrics["gpu_ok"] = "LITERT_CL" in body

    return metrics


def parse_benchmark_metrics_for_session(session_id: str) -> dict:
    """
    Parses LiteRT-LM benchmark metrics from a given DDP session ID.

    Args:
        session_id: The ID of the DeviceRun session to parse.

    Returns:
        A dictionary containing the parsed metrics (prefill, decode, TTFT, peak RAM, GPU status).
    """
    # Define the base directory where the session logs were downloaded.
    base = f"{RESULTS_DIR}/{session_id}"

    # Recursively search for the logcat.txt file within the session's directory structure.
    logs = glob.glob(f"{base}/**/logcat.txt", recursive=True)

    if not logs:
        raise SystemExit(f"No logcat.txt under {base} -- did the job finish?")

    return parse_benchmark_metrics(logs[0])


def get_session_id(output_lines):
    """
    Extracts the session ID from the standard output lines of the gcloud DDP submit command.
    """
    direct_session = None
    for line in output_lines:
        # Look for the exact format where the CLI prints the created session ID
        m = re.search(r"Creating session \[([^\]]+)\]", line)
        if m:
            direct_session = m.group(1)
            break

    return direct_session

def download_session_results(session_id):
    """
    Downloads the artifacts and logs for a given session from GCS to the local Colab environment.
    Note: This function assumes `bucket_id` is defined as a global variable in the notebook.
    """
    # Create the target directory locally
    !mkdir -p {RESULTS_DIR}

    # Copy the session results from the GCS bucket.
    !gcloud storage cp -r gs://{bucket_id}/automation/sessions/{session_id}/ {RESULTS_DIR}/

def print_metrics(device, backend, metrics):
    """
    Helper function to pretty-print the parsed metrics.
    """
    print(f"device   : {device}")
    print(f"backend  : {backend}")
    print(f"prefill  : {metrics['prefill_tok_s']} tokens/sec")
    print(f"decode   : {metrics['decode_tok_s']} tokens/sec")
    print(f"TTFT     : {metrics['ttft_s']} s")
    print(f"peak RAM : {metrics['peak_ram_mb']} MB")

    # Only print GPU delegation status if the backend was set to GPU
    if backend == "gpu":
        print(f"GPU delegated: {metrics['gpu_ok']}")

def generate_dashboard(session_id: str) -> pd.DataFrame:
    """
    Parses the per-device logs for a given session into a table and plots CPU vs GPU.
    Returns the generated pandas DataFrame.
    """
    local_dir = Path(f"{RESULTS_DIR}/{session_id}")
    rows = []

    # rglob recursively finds all .log files in any 'out' directory
    for log in sorted(local_dir.rglob("out/*.log")):
        name = log.stem  # e.g., 'cpu' or 'gpu'

        dev_file = log.with_name("_device.txt")
        exit_file = log.with_name(f"{name}.exit")

        # Read metadata
        device = dev_file.read_text().strip() if dev_file.exists() else re.search(r"job-\d+", str(log)).group()

        # Extract metrics and build row
        row = {
            "device": device,
            "backend": name,
            "exit": exit_file.read_text().strip()
        }
        row.update(parse_benchmark_metrics(str(log)))

        # We only care about OpenCL delegation status for the GPU run
        if name != "gpu":
            row["gpu_ok"] = None

        rows.append(row)

    if not rows:
        print(f"No logs found under {local_dir} -- did the download cell succeed?")
        return None

    # Construct the DataFrame
    df = pd.DataFrame(rows).set_index(["device", "backend"]).sort_index()
    print(df.to_string(float_format=lambda v: f"{v:,.1f}"))

    # Alert on failures or missing GPU delegation
    bad = df[(df.exit != "0") | (df.gpu_ok == False)]
    if not bad.empty:
        print("\nFAILED or no GPU delegation: " + ", ".join(f"{d}/{b}" for d, b in bad.index))

    # Generate the bar charts
    metrics_to_plot = ["prefill_tok_s", "decode_tok_s", "ttft_s", "peak_ram_mb"]
    for metric in metrics_to_plot:
        df[metric].unstack("device").plot.bar(title=metric, figsize=(8, 3), rot=0)

    return df


In [ ]:
# @title Show the direct run's output
# @markdown Downloads `logcat.txt` — the stdout DDP captured — and prints the
# @markdown benchmark summary.

direct_session = get_session_id(direct_output)
if not direct_session:
    raise SystemExit("Could not find a session ID; check the output above.")

download_session_results(direct_session)
metrics = parse_benchmark_metrics_for_session(direct_session)
print_metrics(DIRECT_DEVICE, DIRECT_BACKEND, metrics)


## 🔁 Part 2 — The wrapper script

The direct call runs one configuration. To compare **CPU against GPU** we hand
DDP a shell script instead of the binary, and let the script drive the binary
several times on the device.

That buys three things a single invocation cannot do:

* **Warm-start measurement.** Each configuration runs *twice*. The first pass is
  discarded (`> /dev/null`); it exists only to write the XNNPACK / MLDrift
  caches next to the model. The second pass is the one we keep.
* **Thermal cooldown.** A `sleep 20` between configurations stops the first run
  from throttling the next one and skewing the comparison.
* **Device identification.** DDP labels results `job-000`, `job-001`, … with no
  record of the hardware. `getprop ro.product.model` writes the model name into
  the results so the report can name the device.

Because the script writes log files rather than stdout, this half *does* need
`--paths-to-pull` to bring `/data/local/tmp/out` back.

In [ ]:
%%writefile run_all.sh
#!/system/bin/sh

WORK="/data/local/tmp"
# The OpenCL delegate is loaded at runtime, so it has to be on the loader path.
export LD_LIBRARY_PATH="$WORK"
chmod 755 "$WORK/litert_lm_advanced_main"
rm -rf "$WORK/out" && mkdir -p "$WORK/out"

# DDP reports only job-NNN. Without this the report cannot name the hardware.
getprop ro.product.model > "$WORK/out/_device.txt"

COMMON="--model_path=$WORK/model.litertlm \
    --benchmark=true \
    --benchmark_prefill_tokens=1024 \
    --benchmark_decode_tokens=512 \
    --max_num_tokens=2048 \
    --report_peak_memory_footprint=true"

# run <tag> <extra flags...>
#   First pass warms the caches and is discarded; second pass is measured.
run() {
  tag="$1"; shift
  echo "### $tag"
  "$WORK/litert_lm_advanced_main" $COMMON "$@" >/dev/null 2>&1
  "$WORK/litert_lm_advanced_main" $COMMON "$@" >"$WORK/out/$tag.log" 2>&1
  echo $? > "$WORK/out/$tag.exit"
  sleep 20   # let the SoC cool so the next config is not throttled
}

run cpu --backend=cpu --num_cpu_threads=4
run gpu --backend=gpu

echo "ALL DONE"


In [ ]:
# @title Upload the script and submit the sweep
# @markdown Submits one job per selected device, each running the full
# @markdown CPU + GPU sweep. Uses `--async` so the next cell can report progress.
import uuid

# @markdown The devices to run DDP CLI on;
ddp_target_devices = "pa3q-35,m2q-36,caiman-35" # @param {type: "string"}

# A unique path per run: overwriting a shared script would corrupt any session
# that has not started yet, because DDP fetches the script when the job starts.
run_tag = uuid.uuid4().hex[:8]
SCRIPT_URI = f"gs://{bucket_id}/scripts/{run_tag}/run_all.sh"
!gcloud storage cp run_all.sh "{SCRIPT_URI}"

push = [
    f"{MODEL_URI}={WORK}/model.litertlm",
    f"{LM_BIN}={WORK}/litert_lm_advanced_main",
] + LIB_PUSH

submit_output = !gcloud alpha device-run sessions submit android-executable \
  --project="{ddp_gcp_project}" \
  --location="global" \
  --bucket-name="{bucket_id}" \
  --device="{ddp_target_devices}" \
  --executable="{SCRIPT_URI}" \
  --other-files-to-push="{','.join(push)}" \
  --paths-to-pull="{WORK}/out" \
  --executable-timeout=60m \
  --async 2>&1

print("\n".join(submit_output))


In [ ]:
# @title Poll until the sweep completes, then download the results
# @markdown `sessions wait` blocks silently, so we poll `sessions describe`
# @markdown instead and print progress as jobs finish.
# @markdown
# @markdown ---
# @markdown **Polling interval (seconds):**
import re
import subprocess
import time

POLL_SECONDS = 60 # @param {type:"integer"}

session_id = get_session_id(submit_output)
if not session_id:
    raise SystemExit("Failed to find session ID in the output.")

print(f"Waiting for {session_id}, polling every {POLL_SECONDS}s...\n")

start = time.time()
while True:
    out = subprocess.run(
        ["gcloud", "alpha", "device-run", "sessions", "describe", session_id,
         f"--project={ddp_gcp_project}", "--location=global"],
        capture_output=True, text=True)
    # `describe` writes its status prose to stderr and the job table to stdout.
    status = (out.stderr + out.stdout).strip()

    # Keep just the status and the job tally; drop the long bucket URL.
    keep = [l for l in status.splitlines()
            if l.startswith("Session [") or l.startswith("Job status:")]
    print(f"[{(time.time() - start) / 60:5.1f} min] " + " | ".join(keep))
    if out.returncode != 0:
        print(f"  (describe exited {out.returncode})")

    if "finished with result" in status:
        break
    time.sleep(POLL_SECONDS)


In [ ]:
# @title 📊 Results dashboard
# @markdown Parses the per-device logs into a table and plots CPU vs GPU.
download_session_results(session_id)
generate_dashboard(session_id)